In [7]:
from pyspark.sql import SparkSession

spark = (
SparkSession.builder
.appName("test_avro")
.config("spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,"
        "org.apache.spark:spark-avro_2.13:4.1.1")
.getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

df = (
spark.read.format("kafka") 
.option("kafka.bootstrap.servers", "localhost:9092")
.option("subscribe", "at.vehicle_positions")
.option("startingOffsets", "earliest")
.load()
)
df.count()

8188

In [2]:
df.show(5, truncate=False)

+----------------------------+-------------------------------------------------------------------------------------------------------------------------------+--------------------+---------+------+-----------------------+-------------+
|key                         |value                                                                                                                          |topic               |partition|offset|timestamp              |timestampType|
+----------------------------+-------------------------------------------------------------------------------------------------------------------------------+--------------------+---------+------+-----------------------+-------------+
|[32 35 30 30 30 30 39 32 35]|[00 00 00 00 01 12 32 35 30 30 30 30 39 32 35 07 52 3C 41 75 DE 41 C0 EA 36 BB C8 17 CB 65 40 02 00 00 00 00 D2 F8 92 9C 0D 00]|at.vehicle_positions|0        |0     |2026-03-24 22:50:53.407|0            |
|[33 31 39 33 35 33 30 30 30]|[00 00 00 00 01 12 33 31 39 33

In [3]:
df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)").show(5, truncate=False)

+---------+--------------------------------------------+
|key      |value                                       |
+---------+--------------------------------------------+
|250000925|    250000925\aR<Au�A��6���e@    ����\r |
|319353000|    319353000%]3�fkB��kK\n�e@    ����\r |
|219029326|    219029326��VK'�A���r\r�e@    ����\r |
|512004132|    512004132���lB��JQA��e@]�R=����\r   |
|563303700|    563303700��[�C�����vf@    ����\r  |
+---------+--------------------------------------------+
only showing top 5 rows


In [4]:
from pyspark.sql.avro.functions import from_avro

schema_str = """
{
  "type": "record",
  "name": "VehiclePosition",
  "namespace": "nz.at",
  "fields": [
    {"name": "vehicle_id",  "type": "string"},
    {"name": "latitude",    "type": "double"},
    {"name": "longitude",   "type": "double"},
    {"name": "speed",       "type": ["null", "float"], "default": null},
    {"name": "timestamp",   "type": "long"},
    {"name": "is_deleted",  "type": "boolean", "default": false}
  ]
}
"""

df.select(from_avro("value", schema_str).alias("data")).show(5)

26/03/25 13:12:26 ERROR Executor: Exception in task 0.0 in stage 5.0 (TID 4)
org.apache.spark.SparkException: Malformed records are detected in record parsing. Current parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.
	at org.apache.spark.sql.avro.AvroDataToCatalyst.nullSafeEval(AvroDataToCatalyst.scala:121)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$1(SparkPlan.scala:402)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:901)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.sc

Py4JJavaError: An error occurred while calling o51.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 4) (mac executor driver): org.apache.spark.SparkException: Malformed records are detected in record parsing. Current parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.
	at org.apache.spark.sql.avro.AvroDataToCatalyst.nullSafeEval(AvroDataToCatalyst.scala:121)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$1(Spar

In [5]:
from pyspark.sql.functions import col, substring

df.select(
    col("value").cast("binary"),
    substring(col("value"), 1, 1).alias("magic_byte"),
    substring(col("value"), 2, 4).alias("schema_id_bytes"),
).show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------+----------+---------------+
|value                                                                                                                          |magic_byte|schema_id_bytes|
+-------------------------------------------------------------------------------------------------------------------------------+----------+---------------+
|[00 00 00 00 01 12 32 35 30 30 30 30 39 32 35 07 52 3C 41 75 DE 41 C0 EA 36 BB C8 17 CB 65 40 02 00 00 00 00 D2 F8 92 9C 0D 00]|[00]      |[00 00 00 01]  |
|[00 00 00 00 01 12 33 31 39 33 35 33 30 30 30 25 5D 33 F9 66 6B 42 C0 C8 6B 4B 0A 19 D8 65 40 02 00 00 00 00 8A F8 92 9C 0D 00]|[00]      |[00 00 00 01]  |
|[00 00 00 00 01 12 32 31 39 30 32 39 33 32 36 D2 CE 56 4B 27 EB 41 C0 9E 14 D3 72 0D D0 65 40 02 00 00 00 00 84 F8 92 9C 0D 00]|[00]      |[00 00 00 01]  |
|[00 00 00 00 01 12 35 31 32 30 30 34 31 33 32 1A 93 EB 93

In [6]:
spark.stop()

In [8]:
from pyspark.sql.functions import expr

parsed = df.select(
    col("key").cast("string").alias("key"),
    from_avro(expr("substring(value, 6)"), schema_str).alias("v"),
)
parsed.show(5, truncate=False)

+---------+---------------------------------------------------------------------------------+
|key      |v                                                                                |
+---------+---------------------------------------------------------------------------------+
|250000925|{250000925, -35.73795333333333, 174.34665333333334, 0.0, 1774345769, false}      |
|319353000|{319353000, -36.83908, 174.75305666666668, 0.0, 1774345733, false}               |
|219029326|{219029326, -35.837136666666666, 174.50164166666667, 0.0, 1774345730, false}     |
|512004132|{512004132, -36.84990166666667, 174.80764833333333, 0.0514444, 1774345666, false}|
|563303700|{563303700, -39.472636666666666, 176.92076333333333, 0.0, 1774345779, false}     |
+---------+---------------------------------------------------------------------------------+
only showing top 5 rows


In [9]:
parsed.select("v.latitude", "v.longitude").summary().show()

+-------+-------------------+------------------+
|summary|           latitude|         longitude|
+-------+-------------------+------------------+
|  count|               8188|              8188|
|   mean|-37.780666093792014|174.30170051375782|
| stddev|  2.739884784421429| 5.906657810910348|
|    min|          -48.13442|               0.0|
|    25%|        -37.0692802|       174.5931186|
|    50%|        -36.8725598|       174.7563342|
|    75%|        -36.8158873|174.81041333333334|
|    max|                0.0|178.46394666666666|
+-------+-------------------+------------------+



In [10]:
from pyspark.sql.functions import count, when

parsed.select(
    count("*").alias("total"),
    count("v.speed").alias("with_speed"),
).show()

+-----+----------+
|total|with_speed|
+-----+----------+
| 8188|      8188|
+-----+----------+



In [11]:
from pyspark.sql.functions import from_unixtime

parsed.select(
    from_unixtime(col("v.timestamp")).alias("event_time")
).show(5, truncate=False)

+-------------------+
|event_time         |
+-------------------+
|2026-03-24 22:49:29|
|2026-03-24 22:48:53|
|2026-03-24 22:48:50|
|2026-03-24 22:47:46|
|2026-03-24 22:49:39|
+-------------------+
only showing top 5 rows


In [ ]:
import requests

resp = requests.get("http://localhost:8081/subjects/at.vehicle_positions-value/versions/latest")
sr_schema = resp.json()["schema"]
print(sr_schema)

assert sr_schema.replace(" ", "") == schema_str.replace(" ", "").replace("\n", "")

parsed2 = df.select(
    from_avro(expr("substring(value, 6)"), sr_schema).alias("v"),
)
parsed2.show(5, truncate=False)

{"type":"record","name":"VehiclePosition","namespace":"nz.at","fields":[{"name":"vehicle_id","type":"string"},{"name":"latitude","type":"double"},{"name":"longitude","type":"double"},{"name":"speed","type":["null","float"],"default":null},{"name":"timestamp","type":"long"},{"name":"is_deleted","type":"boolean","default":false}]}
+---------------------------------------------------------------------------------+
|v                                                                                |
+---------------------------------------------------------------------------------+
|{250000925, -35.73795333333333, 174.34665333333334, 0.0, 1774345769, false}      |
|{319353000, -36.83908, 174.75305666666668, 0.0, 1774345733, false}               |
|{219029326, -35.837136666666666, 174.50164166666667, 0.0, 1774345730, false}     |
|{512004132, -36.84990166666667, 174.80764833333333, 0.0514444, 1774345666, false}|
|{563303700, -39.472636666666666, 176.92076333333333, 0.0, 1774345779, false}    